# Merging by Jaccard similarity: an alternative route to channel maps

The exact-cost greedy search (`analytic/mergeUtils.py`) builds a channel map by merging, at every step, the two pattern groups (or the two PSD channels)
whose merge adds the least false-veto cost. This notebook tests a simpler rule: merge the two that are **most similar**, recompute the similarities,
and repeat (`analytic/jaccardMerge.py`). It is shown on the HERD showcase (the thesis selection of 1780 3IR patterns and 1328 PSD tiles) for few
channels, and answers four questions:

1. which Jaccard variant works best, and how it compares with the exact-cost greedy, in cost and in L1 efficiency;
2. how the exact cost evolves along the merge path, and how each Jaccard choice compares with the cheapest merge available at that step;
3. what the merge path looks like, as a tree of merges;
4. what the resulting maps look like, as a network of groups and channels and on the PSD in 3D.

The numbers come from `tools/compareJaccardMerge.py` (about 30 minutes, saved in `examples/results/jaccardComparison.pkl`); this notebook reruns only
two merge paths for the trees (about two minutes).

In [ ]:
import sys, pickle, pathlib
sys.path.insert(0, "../analytic"); sys.path.insert(0, "../stochastic")

import numpy as np
import pandas
import matplotlib.pyplot as plt
import networkx
from scipy.cluster.hierarchy import dendrogram

import patternDump as pd
import losslessUtils as lu
import mergeUtils as mu
import jaccardMerge as jm
import rateUtils as ru
import plotUtils as pu

pu.setStyle()
patterns = pd.selectPatterns(pd.parsePatternDump(pd.defaultDumpPath), **pd.triggerSelections["thesis"])
incidence = pd.buildIncidence(patterns)
M, tileKeys = incidence.matrix, incidence.tileKeys
cost = ru.costMatrix(M.shape[0], ru.tileVetoProbability(tileKeys, window=100e-9))     # 100 ns: Placeholder window (rateUtils.defaultWindow).
referenceCost = float((cost * M).sum())
fitFace = np.array([p.fitSide for p in patterns])
psdFace = np.array([key[0] for key in tileKeys])
comparison = pickle.loads(pathlib.Path("results/jaccardComparison.pkl").read_bytes())
budgets = [(g, c) for g in comparison["groupBudgets"] for c in comparison["channelBudgets"]]
print(f"{M.shape[0]} patterns x {M.shape[1]} tiles; budgets N_G in {comparison['groupBudgets']}, N_C in {comparison['channelBudgets']}")

## 1. The rule

At every step the two most similar groups (or channels) are merged. The similarity of two groups $a$ and $b$ is a weighted Jaccard similarity of their
veto masks, defined as

$$J(a,b) = \frac{\sum_x w_x\,[x\in A]\,[x\in B]}{\sum_x w_x\,[x\in A\cup B]},$$

where

- $A$ and $B$ are the masks of $a$ and $b$: for a group, the set of channels its patterns are vetoed by; for a channel, the set of groups it vetoes;
- $x$ runs over the items of the other side (channels when groups are compared, groups when channels are compared);
- $w_x$ is the weight of item $x$, which is what distinguishes the three variants below;
- $[\,\cdot\,]$ is the indicator, 1 when the condition holds and 0 otherwise.

$J$ is 1 for identical masks and 0 for disjoint ones; the pair with the smallest distance $1-J$ is merged. The three variants are

- **`jaccard`**: $w_x$ is the number of tiles in channel $x$ (or of patterns in group $x$), so that $J$ is the plain Jaccard similarity of the tile
  (or pattern) sets;
- **`costJaccard`**: $w_x$ is the false-veto cost of channel $x$ (or group $x$), the sum over its tiles (patterns) of their mean cost, so that overlap
  is counted in lost photons;
- **`wardJaccard`**: the `costJaccard` distance multiplied by $n_a n_b/(n_a+n_b)$, where $n_a$ and $n_b$ are the cost weights of $a$ and $b$; this
  is the size penalty of Ward's linkage (Ward 1963), which postpones merges of two large clusters.

Routing is the union routing of the exact search, so no RoI tile is ever dropped, and the exact false-veto cost of the current map is carried along
every merge. The groups and channels are merged in one of two orders: all groups first, then the channels (`groupsFirst`), or the reverse
(`channelsFirst`).

## 2. Which variant is best

Exact false-veto cost of every unrefined map, relative to the cost-weighted size of the RoIs ($\sum C M$; lower is better), and the winner per budget.

In [ ]:
names = {method: f"{'exact greedy' if method[0] == 'greedy' else method[0]}, {method[1]}" for method in comparison["cost"]}
costTable = pandas.DataFrame({names[method]: [comparison["cost"][method][budget] for budget in budgets] for method in comparison["cost"]},
                             index=[f"({g}, {c})" for g, c in budgets]).T
jaccardRows = [names[method] for method in comparison["cost"] if method[0] != "greedy"]
winners = costTable.loc[jaccardRows].idxmin()
display(costTable.round(3))
print("Best Jaccard variant per budget:")
print(winners.value_counts().to_string())

The **Ward-scaled variant** is the best overall: it gives the lowest cost at 7 of the 12 budgets (5 of them merging groups first, 2 merging channels
first). The cost-weighted variant wins 4 budgets, three of them at 5 groups by less than 0.004, and plain Jaccard wins one, at (64, 8). With groups
first, `wardJaccard` is within 0.01 of the winner everywhere except (64, 17) and (64, 8), where it trails by 0.09 and 0.05. Merging the channels
first is much worse for the plain and cost-weighted variants: before the groups are merged, a channel's mask spans a thousand groups, and similarity
there says little about cost; the Ward penalty rescues that order in part.

After local refinement of the best map of each family (`mergeUtils.refineLocally`, single tiles and patterns moved while the cost drops), the ratio of
the Jaccard cost to the exact-greedy cost per budget, and the L1 efficiency $\varepsilon_{\rm L1}$ of both on the Geant4 photons ($E^{-2}$ weighting,
accidentals in 100 ns; 340 L0-triggered fiducial photons, the same for both maps):

In [ ]:
groupBudgets, channelBudgets = comparison["groupBudgets"], comparison["channelBudgets"]
ratio = np.array([[comparison["refined"][("jaccard", (g, c))]["cost"] / comparison["refined"][("greedy", (g, c))]["cost"] for c in channelBudgets] for g in groupBudgets])
figure, drawingAxis = plt.subplots(figsize=(7.5, 3.6), constrained_layout=True)
image = drawingAxis.imshow(ratio, cmap="RdBu_r", vmin=0.7, vmax=1.3)
for (row, column), value in np.ndenumerate(ratio):
    drawingAxis.text(column, row, f"{value:.2f}", ha="center", va="center", fontsize=10, color=pu.inkPrimary)
drawingAxis.set(xticks=range(len(channelBudgets)), xticklabels=channelBudgets, yticks=range(len(groupBudgets)), yticklabels=groupBudgets,
                xlabel="PSD channels $N_C$", ylabel="pattern groups $N_G$", title="Refined cost, best Jaccard / exact greedy (below 1: Jaccard better)")
drawingAxis.grid(False)
figure.colorbar(image, ax=drawingAxis, shrink=0.9)
plt.show()
efficiencyTable = pandas.DataFrame({family: [comparison["refined"][(family, budget)]["efficiency"] for budget in budgets] for family in ("greedy", "jaccard")},
                                   index=[f"({g}, {c})" for g, c in budgets]).T
efficiencyTable.index = ["eps_L1, exact greedy refined", "eps_L1, best Jaccard refined"]
display(efficiencyTable.round(3))

Jaccard is 5 to 32% costlier at 64 and 16 groups, and about 1% cheaper at 5 groups with 8 to 32 channels: the exact greedy is not optimal there
either. The efficiencies agree within 0.01 at most budgets; at 4 channels and at (5, 8) the Jaccard maps lose 0.01 to 0.02. At (5, 8) the Jaccard map
has the lower cost but the lower efficiency: the cost is a proxy (backsplash averaged over triggers, equal pattern weights), and replaying the photons
event by event disagrees with it at this level.

## 3. Progress along the optimisation

Left: the exact cost after every merge, from the lossless map (cost 0) to (5, 8), for every method and order. Right: every 25 merges of the
`wardJaccard` groups-first path, the cost added by the Jaccard choice divided by the cost of the cheapest merge available from the same state
(the exact greedy's choice there), for the group merges; the channel merges checked were all free for both.

In [ ]:
colours = dict(zip(["greedy"] + list(jm.variants), pu.categoricalColours))
figure, (trajectoryAxis, regretAxis) = plt.subplots(1, 2, figsize=(15, 5.2), constrained_layout=True)
for (method, order), values in comparison["trajectories"].items():
    trajectoryAxis.plot(np.arange(1, values.size + 1), values, color=colours[method], linestyle="-" if order == "groupsFirst" else "--", linewidth=2,
                        label=f"{'exact greedy' if method == 'greedy' else method}, {order}")
trajectoryAxis.set_yscale("symlog", linthresh=0.01)
trajectoryAxis.set(xlabel="merges from the lossless map", ylabel=r"exact false-veto cost / $\sum C M$", title="Exact cost along each merge path to (5, 8)")
trajectoryAxis.legend(fontsize=8, loc="upper left")
checks = [(step, added / cheapest) for step, side, added, cheapest in comparison["regret"] if cheapest > 1e-12]
steps, ratios = np.array(checks).T
regretAxis.plot(steps, ratios, "o", color=colours["wardJaccard"], markersize=6)
regretAxis.axhline(1.0, color=pu.inkPrimary, linewidth=1)
regretAxis.set(yscale="log", xlabel="merges from the lossless map (group phase)", ylabel="added by the Jaccard choice / cheapest available",
               title="Each checked Jaccard merge against the cheapest")
plt.show()
print(f"{len(ratios)} checks with a cost: the Jaccard choice was the cheapest in {np.mean(ratios <= 1 + 1e-9):.0%}; "
      f"median ratio {np.median(ratios):.2f}, quartiles {np.percentile(ratios, 25):.2f} and {np.percentile(ratios, 75):.2f}")

The Jaccard path is locally wasteful (its median merge costs several times the cheapest one) and still ends below the exact greedy at 5 groups:
taking the cheapest merge at every step does not give the cheapest path. In the groups-first order the cost is all in the group phase and in the
last few channel merges: with 5 groups the 1328 tiles fall into at most 17 distinct channel masks, so the channel merges down to 17 are free.

## 4. The merge path as a tree

A dendrogram draws an agglomeration as a tree: each join is one merge, at a height equal to the exact cost after it (non-decreasing, since a merge
under union routing can only add tiles to masks). The full trees have about 1100 leaves (the lossless groups) and 1300 (the lossless channels), so
only the last merges are drawn; each leaf of the drawing is a cluster, labelled by its one or two main FIT faces (PSD faces for channels; "+" when more faces are present) and its size. The paths are rerun to a
single group (or channel) so that the tree is complete; up to the dashed cut they are the same paths as to (5, 8).

In [ ]:
lossless = mu._losslessWithLabels(M, None, None)
numberOfLosslessGroups, numberOfLosslessChannels = lossless.routing.shape

def faceSummary(faces):
    names, counts = np.unique(faces, return_counts=True)
    order = np.argsort(-counts)
    return " ".join(f"{names[k]}" for k in order[:2]) + (" +" if len(names) > 2 else "")

def drawTree(drawingAxis, trace, side, numberOfItems, cutClusters, shownClusters, title):
    linkage, members = jm.linkageFromTrace(trace, side, numberOfItems)
    leavesOf = lossless.rowGroup if side == "group" else lossless.colGroup             # Lossless item of every pattern (tile).
    facesOf = fitFace if side == "group" else psdFace
    unit = "patterns" if side == "group" else "tiles"
    def label(cluster):
        rows = np.isin(leavesOf, members[cluster])
        return f"{faceSummary(facesOf[rows])} ({rows.sum()} {unit})"
    cutHeight = 0.5 * (linkage[-cutClusters, 2] + linkage[-cutClusters + 1, 2]) / referenceCost
    relative = linkage.copy(); relative[:, 2] /= referenceCost
    dendrogram(relative, ax=drawingAxis, truncate_mode="lastp", p=shownClusters, leaf_label_func=label, leaf_rotation=90, leaf_font_size=8,
               color_threshold=cutHeight, above_threshold_color=pu.inkMuted)
    drawingAxis.axhline(cutHeight, color=pu.inkPrimary, linestyle="--", linewidth=1)
    drawingAxis.annotate(f"cut: {cutClusters} {side}s", (drawingAxis.get_xlim()[1], cutHeight), xytext=(-4, 4), textcoords="offset points",
                         ha="right", fontsize=9, color=pu.inkPrimary)
    drawingAxis.set_yscale("symlog", linthresh=0.01)
    drawingAxis.set(ylabel=r"exact cost after the merge / $\sum C M$", title=title)

groupTraces = {variant: jm.mergeByJaccard(M, 1, numberOfLosslessChannels, cost=cost, variant=variant, order="groupsFirst")[1] for variant in ("wardJaccard", "jaccard")}
figure, axes = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)
for drawingAxis, (variant, trace) in zip(axes, groupTraces.items()):
    drawTree(drawingAxis, trace, "group", numberOfLosslessGroups, 5, 24, f"Group merges, {variant} (last 23 merges of {len(trace)})")
plt.show()

At the 5-group cut the `wardJaccard` clusters follow the FIT faces (with a few TOP patterns joined to Xneg and Ypos), found without being told about
faces, and its tree is balanced: the leaves drawn hold 20 to 160 patterns. Plain Jaccard builds a lopsided tree: four clusters of 260 to 400 patterns
grow early and absorb stragglers of 2 to 30 patterns only at the end. At 5 groups both end at almost the same cost (1.268 and 1.269 at (5, 32)), but
the lopsided tree is what costs plain Jaccard at 16 and 64 groups (1.19 against 0.94 at (16, 32), 1.06 against 0.73 at (64, 32)).

The channel merges of the same `wardJaccard` path, after the groups have reached 5 (all channel merges down to 17 are free, so the drawing starts at
17 clusters; the grey leaves are single clusters still apart at the 8-channel cut):

In [ ]:
channelTrace = jm.mergeByJaccard(M, 5, 1, cost=cost, variant="wardJaccard", order="groupsFirst")[1]
figure, drawingAxis = plt.subplots(figsize=(11, 5.5), constrained_layout=True)
drawTree(drawingAxis, channelTrace, "channel", numberOfLosslessChannels, 8, 17, "Channel merges after the groups, wardJaccard, groups first")
plt.show()

## 5. The resulting maps

The refined maps at (5, 8), best Jaccard (seeded by `wardJaccard`, groups first) and exact greedy, as networks: pattern groups on the left (labelled by
their FIT faces and size), channels on the right (labelled by their PSD faces and tile count), and an edge where the channel vetoes the group.

In [ ]:
maps = {"best Jaccard": comparison["refined"][("jaccard", (5, 8))]["map"], "exact greedy": comparison["refined"][("greedy", (5, 8))]["map"]}

def drawNetwork(drawingAxis, channelMap, title):
    graph = networkx.Graph()
    groupNodes = [f"G{k}" for k in range(channelMap.numberOfGroups)]
    channelNodes = [f"C{l}" for l in range(channelMap.numberOfChannels)]
    graph.add_nodes_from(groupNodes); graph.add_nodes_from(channelNodes)
    graph.add_edges_from((f"G{k}", f"C{l}") for k, l in zip(*np.nonzero(channelMap.routing)))
    position = {node: (0.0, -index) for index, node in enumerate(groupNodes)}
    position.update({node: (1.0, -index * (len(groupNodes) - 1) / max(len(channelNodes) - 1, 1)) for index, node in enumerate(channelNodes)})
    groupSize = np.bincount(channelMap.rowGroup, minlength=channelMap.numberOfGroups)
    channelSize = np.bincount(channelMap.colGroup, minlength=channelMap.numberOfChannels)
    networkx.draw_networkx_edges(graph, position, ax=drawingAxis, edge_color=pu.inkMuted, width=1.2, alpha=0.7)
    networkx.draw_networkx_nodes(graph, position, nodelist=groupNodes, ax=drawingAxis, node_color=pu.categoricalColours[0], node_size=40 + groupSize)
    networkx.draw_networkx_nodes(graph, position, nodelist=channelNodes, ax=drawingAxis, node_color=pu.categoricalColours[1], node_size=40 + channelSize)
    for k, node in enumerate(groupNodes):
        drawingAxis.text(position[node][0] - 0.06, position[node][1], f"{faceSummary(fitFace[channelMap.rowGroup == k])} ({groupSize[k]})",
                         ha="right", va="center", fontsize=9, color=pu.inkPrimary)
    for l, node in enumerate(channelNodes):
        drawingAxis.text(position[node][0] + 0.06, position[node][1], f"C{l}: {faceSummary(psdFace[channelMap.colGroup == l])} ({channelSize[l]})",
                         ha="left", va="center", fontsize=9, color=pu.inkPrimary)
    mapCost = mu.computeMaskCost(M, channelMap, cost)[0] / referenceCost
    drawingAxis.set_title(f"{title}: cost {mapCost:.3f}", color=pu.inkPrimary)
    drawingAxis.set_xlim(-0.9, 1.9); drawingAxis.axis("off")

figure, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
for drawingAxis, (name, channelMap) in zip(axes, maps.items()):
    drawNetwork(drawingAxis, channelMap, f"{name}, (5, 8), refined")
plt.show()

On the PSD, the same two maps, one colour per channel, seen from two sides so that all five faces show (layer 0 drawn; the legend gives the FIT faces
whose patterns each channel vetoes):

In [ ]:
from matplotlib.patches import Patch
figure = plt.figure(figsize=(17, 11), constrained_layout=True)
for row, (name, channelMap) in enumerate(maps.items()):
    groupFaces = [faceSummary(fitFace[channelMap.rowGroup == k]) for k in range(channelMap.numberOfGroups)]
    colourOf = {(face, tileId): pu.categoricalColours[channelMap.colGroup[column]]
                for column, (face, layer, tileId) in enumerate(tileKeys) if layer == 0}
    for column, azimuth in enumerate((-60, 120)):
        pu.plotPsdCube(figure.add_subplot(2, 3, 3 * row + column + 1, projection="3d"), colourOf, azimuth=azimuth,
                       title=f"{name}, (5, 8), refined, azimuth {azimuth} deg")
    legendAxis = figure.add_subplot(2, 3, 3 * row + 3); legendAxis.axis("off")
    legendAxis.legend(handles=[Patch(facecolor=pu.categoricalColours[l], label=f"C{l} vetoes " + ", ".join(groupFaces[k] for k in np.flatnonzero(channelMap.routing[:, l])))
                               for l in range(channelMap.numberOfChannels)], loc="center left", fontsize=9, frameon=False, title=name)
plt.show()

## 6. Conclusions

- **Best variant**: `wardJaccard`, best at 7 of 12 budgets and close at most others with groups first. The size penalty matters: without it, a few
  large groups grow early, which costs at 16 and 64 groups.
- **Against the exact greedy**: a few percent worse with many groups, about 1% better at 5 groups; as fast as it is (a scan over 12 budgets in about
  a minute, the greedy in about four), it is a useful second seed for the refinement and for the stochastic stage, not a replacement.
- **Progress**: locally the Jaccard choices are rarely the cheapest, yet the path can end lower; greedy decisions on the exact cost are not optimal
  either, which is one more reason for the stochastic stage, seeded by both.